# 05_catboost

## Роль ноутбука

CatBoost здесь выступает как сильная глобальная модель, которая:
- сравнивается с SARIMA на `top_pairs`;
- затем может быть обучена на полном train;
- формирует финальный recursive submission.

## Что читает

- `artifacts/datasets/train_processed.csv`
- `artifacts/datasets/test_processed.csv`
- `artifacts/datasets/top_pairs.csv`
- `artifacts/metadata/splits.json`

## Что сохраняет

- `artifacts/params/catboost_best_params.json`
- `artifacts/metrics/catboost_store_metrics_by_split_recursive.csv`
- `artifacts/predictions/catboost_store_preds_by_split_recursive.pkl`
- `artifacts/submissions/submission_catboost_recursive.csv`


## Методологическая оговорка

Чтобы не превращать подбор гиперпараметров в многодневный расчет,
Optuna ниже использует быстрый proxy-режим на первом сплите и на `top_pairs`.
Официальная оценка качества для главы 3 при этом считается отдельно,
через recursive forecast на всех сплитах.


In [ ]:
import os
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

from catboost import CatBoostRegressor, Pool
import optuna

import warnings
warnings.filterwarnings("ignore")

# Настройка pandas
pd.set_option("display.max_rows", 50)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Логгер
import logging

logger = logging.getLogger("catboost_store_level")
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)

logger.info("Logger catboost_store_level initialized")


In [ ]:
PROJECT_ROOT = Path('..')
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'
DATASETS_DIR = ARTIFACTS_DIR / 'datasets'
METADATA_DIR = ARTIFACTS_DIR / 'metadata'
METRICS_DIR = ARTIFACTS_DIR / 'metrics'
PREDICTIONS_DIR = ARTIFACTS_DIR / 'predictions'
PARAMS_DIR = ARTIFACTS_DIR / 'params'
SUBMISSIONS_DIR = ARTIFACTS_DIR / 'submissions'

for path in [METRICS_DIR, PREDICTIONS_DIR, PARAMS_DIR, SUBMISSIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

train_processed = pd.read_csv(DATASETS_DIR / 'train_processed.csv', parse_dates=['date'])
test_processed = pd.read_csv(DATASETS_DIR / 'test_processed.csv', parse_dates=['date'])
top_pairs_df = pd.read_csv(DATASETS_DIR / 'top_pairs.csv')
with open(METADATA_DIR / 'splits.json', 'r', encoding='utf-8') as f:
    splits_json = json.load(f)

splits = []
for s in splits_json:
    splits.append({
        'name': s['name'],
        'train_idx': pd.Index(s['train_idx']),
        'val_idx': pd.Index(s['val_idx']),
        'train_end': pd.to_datetime(s['train_end']),
        'val_start': pd.to_datetime(s['val_start']),
        'val_end': pd.to_datetime(s['val_end']),
    })

top_pairs = set(top_pairs_df[['store_nbr', 'family']].itertuples(index=False, name=None))
logger.info(f'Loaded train_processed: {train_processed.shape}')
logger.info(f'Loaded test_processed: {test_processed.shape}')
logger.info(f'Loaded top_pairs: {len(top_pairs)}')


## Политика признаков

- `sales_sum` исключен из базового списка признаков как потенциальная утечка;
- безопасный submission-режим использует признаки, доступные и в train, и в test;
- дополнительные признаки допустимы только как осознанный research-mode.


In [ ]:
def add_time_features(df):
    """
    Добавляет базовые временные признаки в df:
    - dayofweek, weekofyear, month, day, is_weekend.
    Ожидает колонку 'date' типа datetime64[ns].
    """
    df = df.copy()
    df["dow"] = df["date"].dt.weekday.astype("int8")
    df["weekofyear"] = df["date"].dt.isocalendar().week.astype("int16")
    df["month"] = df["date"].dt.month.astype("int8")
    df["day"] = df["date"].dt.day.astype("int8")
    df["is_weekend"] = df["dow"].isin([5, 6]).astype("int8")
    return df


In [ ]:
def add_lag_features(df, lags, group_cols=["store_nbr", "family"], target_col="sales"):
    """
    Добавляет лаги target_col на заданные сдвиги в днях.
    """
    df = df.sort_values(["date"] + group_cols).copy()
    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = (
            df.groupby(group_cols)[target_col]
            .shift(lag)
        )
    return df

def add_rolling_features(df, windows, group_cols=["store_nbr", "family"], target_col="sales"):
    df = df.sort_values(["date"] + group_cols).copy()
    g = df.groupby(group_cols)[target_col]
    for w in windows:
        df[f"{target_col}_rolling_mean_{w}"] = (
            g.shift(1).rolling(window=w, min_periods=1).mean()
        )
    return df


In [ ]:
# Список категориальных признаков для CatBoost
CAT_FEATURES = [
    "store_nbr",
    "family",
    "city",
    "state",
    "store_type",
    "cluster",
    "type",
    "locale",
    "agg_description",
]

# Бинарные/индикаторные признаки
BIN_FEATURES = [
    "is_holiday",
    "is_payday",
    "transferred",
]

# Базовые безопасные численные признаки для общего и submission-режима
SAFE_NUM_FEATURES_BASE = [
    "onpromotion",
]

# Эти признаки можно возвращать только после отдельной проверки доступности
# на тестовом горизонте и отсутствия методологических проблем.
OPTIONAL_RESEARCH_NUM_FEATURES = [
    col for col in ["transactions", "dcoilwtico"]
    if col in train_processed.columns and col in test_processed.columns
]

NUM_FEATURES_BASE = SAFE_NUM_FEATURES_BASE + OPTIONAL_RESEARCH_NUM_FEATURES

# Временные фичи, которые добавим
TIME_FEATURES = ["dow", "weekofyear", "month", "day", "is_weekend"]


In [ ]:
def make_train_val_for_split(
    split,
    full_df,
    top_pairs,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Готовит X_train, y_train, X_val, y_val для заданного сплита.

    Параметры:
    - split: словарь из splits (name, train_idx, val_idx, ...)
    - full_df: train_processed
    - top_pairs: множество (store_nbr, family)
    - lags: tuple лагов для sales
    - roll_windows: окна для скользящего среднего по sales

    Возвращает:
    - X_train, y_train, X_val, y_val, cat_features
    """
    split_name = split["name"]
    logger.info(f"Preparing data for {split_name}")

    # Берём train/val по индексам
    train_df = full_df.loc[split["train_idx"]].copy()
    val_df   = full_df.loc[split["val_idx"]].copy()

    # Фильтрация по top_pairs
    def _filter(df):
        mask = df[["store_nbr", "family"]].apply(tuple, axis=1).isin(top_pairs)
        return df[mask].copy()

    train_df = _filter(train_df)
    val_df   = _filter(val_df)

    logger.info(
        f"{split_name} | train rows after top_pairs filter: {len(train_df)}, "
        f"val rows: {len(val_df)}"
    )

    # Склеиваем train+val для корректного расчёта лагов (история перед валидацией)
    combined = pd.concat([train_df, val_df], axis=0).sort_values(["store_nbr", "family", "date"])

    # Добавляем временные фичи
    combined = add_time_features(combined)

    # Лаги по sales
    combined = add_lag_features(combined, lags=lags, group_cols=["store_nbr", "family"], target_col="sales")

    # Скользящие средние
    combined = add_rolling_features(combined, windows=roll_windows, group_cols=["store_nbr", "family"], target_col="sales")

    # После лагов/rolling первые дни ряда будут с NaN — отрежем их
    # Но делаем это аккуратно: только там, где NaN в хотя бы одном из lag/rolling.
    feature_cols_lag_roll = [col for col in combined.columns if "lag_" in col or "rolling_mean_" in col]
    combined = combined.dropna(subset=feature_cols_lag_roll)

    # Разделяем обратно на train/val по дате
    train_end = train_df["date"].max()
    val_start = val_df["date"].min()
    val_end   = val_df["date"].max()

    train_mask = (combined["date"] <= train_end)
    val_mask   = (combined["date"] >= val_start) & (combined["date"] <= val_end)

    train_fe = combined[train_mask].copy()
    val_fe   = combined[val_mask].copy()

    logger.info(
        f"{split_name} | after lag/rolling dropna: train={len(train_fe)}, val={len(val_fe)}"
    )

    # Целевая переменная
    y_train = train_fe["sales"].astype("float32")
    y_val   = val_fe["sales"].astype("float32")

    # Формируем список всех фичей
    feature_cols = (
        NUM_FEATURES_BASE
        + TIME_FEATURES
        + feature_cols_lag_roll
        + BIN_FEATURES
        + CAT_FEATURES
    )

    # Оставляем только существующие в датафрейме
    feature_cols = [c for c in feature_cols if c in train_fe.columns]

    X_train = train_fe[feature_cols].copy()
    X_val   = val_fe[feature_cols].copy()

    # CatBoost требует индексы категориальных колонок
    cat_features = [i for i, c in enumerate(feature_cols) if c in CAT_FEATURES + BIN_FEATURES]

    logger.info(
        f"{split_name} | X_train shape={X_train.shape}, X_val shape={X_val.shape}, "
        f"n_cat_features={len(cat_features)}"
    )

    return X_train, y_train, X_val, y_val, feature_cols, cat_features


In [ ]:
def rmsle_metric(y_true, y_pred):
    """
    RMSLE в исходном масштабе (после обратного expm1).
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    y_pred = np.clip(y_pred, 0, None)
    return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))


In [ ]:
optuna_split = splits[0]
optuna_X_train, optuna_y_train, optuna_X_val, optuna_y_val, optuna_feature_cols, optuna_cat_features = make_train_val_for_split(
    optuna_split,
    train_processed,
    top_pairs,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
)
logger.info(f'Optuna proxy split: {optuna_split["name"]}')
logger.info(f'Optuna train shape: {optuna_X_train.shape}, val shape: {optuna_X_val.shape}')


In [ ]:
def objective(trial):
    param = {
        'depth': trial.suggest_int('depth', 4, 10),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-3, 10.0, log=True),
        'bagging_temperature': trial.suggest_float('bagging_temperature', 0.0, 5.0),
        'border_count': trial.suggest_int('border_count', 64, 255),
    }

    y_train_log = np.log1p(optuna_y_train)
    y_val_log = np.log1p(optuna_y_val)

    train_pool = Pool(data=optuna_X_train, label=y_train_log, cat_features=optuna_cat_features)
    val_pool = Pool(data=optuna_X_val, label=y_val_log, cat_features=optuna_cat_features)

    model = CatBoostRegressor(
        loss_function='RMSE',
        eval_metric='RMSE',
        depth=param['depth'],
        learning_rate=param['learning_rate'],
        l2_leaf_reg=param['l2_leaf_reg'],
        bagging_temperature=param['bagging_temperature'],
        border_count=param['border_count'],
        random_state=42,
        thread_count=4,
        task_type='CPU',
        verbose=False,
        od_type='Iter',
        od_wait=50,
        iterations=3000,
    )

    model.fit(train_pool, eval_set=val_pool, use_best_model=True, verbose=False)
    y_pred_log = model.predict(val_pool)
    y_pred = np.expm1(y_pred_log)
    score = rmsle_metric(optuna_y_val, y_pred)
    logger.info(f'Trial {trial.number}: params={param}, RMSLE={score:.4f}')
    return score


## Proxy-настройка гиперпараметров

Эта часть нужна не для финальной отчетной метрики, а для воспроизводимого и
относительно экономичного подбора параметров перед recursive evaluation.


In [ ]:
import multiprocessing

n_cores = multiprocessing.cpu_count()
logger.info(f"CPU cores available: {n_cores}")

study = optuna.create_study(
    direction="minimize",
    study_name="catboost_rmsle_log1p",
)

N_TRIALS = 20  # можно увеличить/уменьшить по ситуации

logger.info(f"Starting Optuna optimization with {N_TRIALS} trials...")

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=2,  # параллель по всем ядрам
    show_progress_bar=True,
)

logger.info(f"Best RMSLE: {study.best_value:.4f}")
logger.info(f"Best params: {study.best_params}")


In [ ]:
best_params = study.best_params
best_params['n_trials'] = N_TRIALS
best_params['best_value'] = study.best_value
params_path = PARAMS_DIR / 'catboost_best_params.json'
with open(params_path, 'w', encoding='utf-8') as f:
    json.dump(best_params, f, indent=2, ensure_ascii=False)
logger.info(f'Best CatBoost params saved to {params_path}')
best_params


## Recursive CV на всех сплитах

Это главный блок для сопоставления CatBoost с SARIMA и остальными моделями.
Именно его метрики нужно использовать в сравнительной таблице главы 3.


In [ ]:
def calculate_metrics_all(y_true, y_pred):
    """
    Возвращает dict с RMSLE, RMSE, MAE, WAPE.
    """
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    # Обрезаем прогнозы по нулю
    y_pred = np.clip(y_pred, 0, None)

    # RMSLE
    rmsle = np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

    # RMSE
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))

    # MAE
    mae = np.mean(np.abs(y_true - y_pred))

    # WAPE
    denom = np.sum(np.abs(y_true))
    wape = np.nan if denom == 0 else np.sum(np.abs(y_true - y_pred)) / denom

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "WAPE": wape,
    }


In [ ]:
def fit_catboost_on_split(
    split,
    full_df,
    top_pairs,
    best_params,
    thread_count=4,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Обучает CatBoost на train-части сплита.
    Ничего не знает про валидацию и метрики — только fit.

    Возвращает:
    - model: обученный CatBoostRegressor
    - feature_cols: список фичей в том порядке, как их ждёт модель
    - cat_features_idx: индексы категориальных фичей в feature_cols
    """
    split_name = split["name"]
    logger.info(f"[{split_name}] Fitting CatBoost (train only)")

    # Берём старую утилиту, но будем использовать только train-часть
    X_train, y_train, X_val_dummy, y_val_dummy, feature_cols, cat_features_idx = make_train_val_for_split(
        split,
        full_df,
        top_pairs,
        lags=lags,
        roll_windows=roll_windows,
    )

    # Лог-преобразование таргета
    y_train_log = np.log1p(y_train)

    train_pool = Pool(
        data=X_train,
        label=y_train_log,
        cat_features=cat_features_idx,
    )

    model = CatBoostRegressor(
        loss_function="RMSE",
        eval_metric="RMSE",
        depth=best_params["depth"],
        learning_rate=best_params["learning_rate"],
        l2_leaf_reg=best_params["l2_leaf_reg"],
        bagging_temperature=best_params["bagging_temperature"],
        border_count=best_params["border_count"],
        random_state=42,
        thread_count=thread_count,
        task_type="CPU",
        verbose=False,
        od_type="Iter",
        od_wait=50,
        iterations=2000,  # можно потом уменьшить
    )

    start_time = datetime.now()
    logger.info(f"[{split_name}] CatBoost fit started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

    model.fit(
        train_pool,
        use_best_model=False,  # в этой функции только train, без val
        verbose=False,
    )

    end_time = datetime.now()
    elapsed = (end_time - start_time).total_seconds()
    logger.info(f"[{split_name}] CatBoost fit finished in {elapsed:.1f} seconds")

    return model, feature_cols, cat_features_idx


In [ ]:
def prepare_history_and_future(split, full_df, top_pairs):
    """
    Готовит:
    - history_df: train-часть сплита (только top_pairs), отсортированную по (store_nbr, family, date);
    - future_df: val-часть сплита (только top_pairs), также отсортированную;
    - y_true: вектор истинных продаж для future_df (в том же порядке).
    """
    split_name = split["name"]

    train_df = full_df.loc[split["train_idx"]].copy()
    val_df   = full_df.loc[split["val_idx"]].copy()

    def _filter(df):
        mask = df[["store_nbr", "family"]].apply(tuple, axis=1).isin(top_pairs)
        return df[mask].copy()

    train_df = _filter(train_df)
    val_df   = _filter(val_df)

    logger.info(
        f"[{split_name}] history rows (train, top_pairs)={len(train_df)}, "
        f"future rows (val, top_pairs)={len(val_df)}"
    )

    history_df = train_df.sort_values(["store_nbr", "family", "date"]).copy()
    future_df  = val_df.sort_values(["store_nbr", "family", "date"]).copy()

    y_true = future_df["sales"].astype("float32").values

    # В future_df 'sales' будем перезаписывать предсказаниями, поэтому сохраним y_true отдельно.
    future_df = future_df.drop(columns=["sales"])

    return history_df, future_df, y_true


In [ ]:
def recursive_forecast_catboost_on_split(
    split,
    full_df,
    top_pairs,
    model,
    feature_cols,
    cat_features_idx,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Делает рекурсивный прогноз CatBoost на 16-дневном val-окне сплита.

    Шаги:
    - берём history_df (train) и future_df (val) из prepare_history_and_future;
    - для каждого дня val-окна:
        * добавляем его «каркас» (без sales) к истории;
        * пересчитываем временные, лаговые и rolling-фичи;
        * предсказываем sales на этот день;
        * записываем предсказания в историю, чтобы они участвовали в лаговых фичах следующего дня.
    Возвращает:
    - val_pred_df: DataFrame с колонками [date, store_nbr, family, y_true, y_pred]
    """
    split_name = split["name"]
    logger.info(f"[{split_name}] Recursive CatBoost forecast on validation window")

    history_df, future_df, y_true = prepare_history_and_future(split, full_df, top_pairs)

    # Будем накапливать историю с предсказаниями
    hist = history_df.copy()

    # Список предсказаний по дням
    preds_frames = []

    # Уникальные даты валидации по возрастанию
    val_dates = sorted(future_df["date"].unique())

    for cur_date in val_dates:
        logger.info(f"[{split_name}] Forecasting date {cur_date}")

        # Строки для текущего дня (exogenous features, без sales)
        day_future = future_df[future_df["date"] == cur_date].copy()

        # Временно добавляем пустой sales (NaN), чтобы лаги считались от hist
        day_future["sales"] = np.nan

        # Склеиваем историю + текущий день
        combined = pd.concat([hist, day_future], axis=0, ignore_index=True)

        # Фичи: время, лаги, rolling
        combined = add_time_features(combined)
        combined = add_lag_features(
            combined,
            lags=lags,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )
        combined = add_rolling_features(
            combined,
            windows=roll_windows,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )

        # Берём только текущий день уже с фичами
        feature_cols_lag_roll = [c for c in combined.columns if "lag_" in c or "rolling_mean_" in c]
        cur_df = combined[combined["date"] == cur_date].copy()

        # Отбрасываем строки, где нет необходимых лагов/rolling
        cur_df = cur_df.dropna(subset=feature_cols_lag_roll)

        # Если вдруг всё отвалилось (нехватка истории) — пропускаем с предупреждением
        if cur_df.empty:
            logger.warning(f"[{split_name}] No rows to predict for {cur_date}, skipping")
            continue

        # Готовим X для модели в том же порядке feature_cols
        X_cur = cur_df[feature_cols].copy()

        pool_cur = Pool(
            data=X_cur,
            cat_features=cat_features_idx,
        )

        # Предсказания в лог-пространстве → назад в исходный масштаб
        y_pred_log = model.predict(pool_cur)
        y_pred = np.expm1(y_pred_log)

        cur_df["y_pred"] = y_pred

        # Истинные продажи для этого дня берём из full_df
        true_day = full_df[
            (full_df["date"] == cur_date)
            & (full_df[["store_nbr", "family"]]
               .apply(tuple, axis=1)
               .isin(top_pairs))
        ][["store_nbr", "family", "sales"]].rename(columns={"sales": "y_true"})

        cur_df = cur_df.merge(
            true_day,
            on=["store_nbr", "family"],
            how="left",
        )

        preds_frames.append(
            cur_df[["date", "store_nbr", "family", "y_true", "y_pred"]]
        )

        # Обновляем историю: для следующего дня 'sales' должны быть равны предсказанным y_pred
        # Берём ровно те строки, которые мы сейчас спрогнозировали (по ключам)
        hist_update = cur_df[["date", "store_nbr", "family"]].copy()
        hist_update["sales"] = y_pred

        # Также нужны остальные признаки (onpromotion, city, state, ...),
        # поэтому мёрджим их из future_df
        hist_update = hist_update.merge(
            future_df[future_df["date"] == cur_date].drop(columns=[]),
            on=["date", "store_nbr", "family"],
            how="left",
        )

        hist = pd.concat([hist, hist_update], axis=0, ignore_index=True)

    # Собираем всё вместе
    val_pred_df = pd.concat(preds_frames, axis=0, ignore_index=True)

    return val_pred_df


In [ ]:
def evaluate_catboost_on_split_recursive(
    split,
    full_df,
    top_pairs,
    model,
    feature_cols,
    cat_features_idx,
):
    """
    Обёртка: рекурсивный прогноз + расчёт метрик на сплите.
    Ответственность: только оценка качества.
    """
    split_name = split["name"]

    val_pred_df = recursive_forecast_catboost_on_split(
        split=split,
        full_df=full_df,
        top_pairs=top_pairs,
        model=model,
        feature_cols=feature_cols,
        cat_features_idx=cat_features_idx,
    )

    y_true = val_pred_df["y_true"].values
    y_pred = val_pred_df["y_pred"].values

    metrics = calculate_metrics_all(y_true, y_pred)
    metrics["split"] = split_name
    metrics["model"] = "catboost_store_family_recursive"

    logger.info(
        f"[{split_name}] CatBoost (recursive) metrics: "
        f"RMSLE={metrics['RMSLE']:.4f}, "
        f"RMSE={metrics['RMSE']:.2f}, "
        f"MAE={metrics['MAE']:.2f}, "
        f"WAPE={metrics['WAPE']:.4f}"
    )

    return metrics, val_pred_df


In [ ]:
catboost_metrics_by_split = {}
catboost_preds_by_split = {}

THREAD_COUNT = -1

for s in splits:
    split_name = s["name"]
    logger.info(f"=== {split_name} | CatBoost recursive 16-day forecast ===")

    # 1. Обучение
    model, feature_cols, cat_features_idx = fit_catboost_on_split(
        split=s,
        full_df=train_processed,
        top_pairs=top_pairs,
        best_params=best_params,
        thread_count=THREAD_COUNT,
    )

    # 2. Рекурсивный прогноз + метрики
    metrics, val_pred_df = evaluate_catboost_on_split_recursive(
        split=s,
        full_df=train_processed,
        top_pairs=top_pairs,
        model=model,
        feature_cols=feature_cols,
        cat_features_idx=cat_features_idx,
    )

    catboost_metrics_by_split[split_name] = metrics
    catboost_preds_by_split[split_name] = val_pred_df


In [ ]:
rows = []
for split_name, m in catboost_metrics_by_split.items():
    rows.append({
        'split': split_name,
        'model': m.get('model', 'catboost_store_family_recursive'),
        'RMSLE': m['RMSLE'],
        'RMSE': m['RMSE'],
        'MAE': m['MAE'],
        'WAPE': m['WAPE'],
    })

catboost_metrics_df = pd.DataFrame(rows).set_index('split')
mean_row = pd.DataFrame(catboost_metrics_df.mean(axis=0).to_dict(), index=['mean'])
catboost_metrics_df = pd.concat([catboost_metrics_df, mean_row])
metrics_path = METRICS_DIR / 'catboost_store_metrics_by_split_recursive.csv'
catboost_metrics_df.to_csv(metrics_path, index=True)
logger.info(f'CatBoost recursive-forecast metrics saved to {metrics_path}')

preds_path = PREDICTIONS_DIR / 'catboost_store_preds_by_split_recursive.pkl'
with open(preds_path, 'wb') as f:
    pickle.dump(catboost_preds_by_split, f)
logger.info(f'CatBoost predictions by split saved to {preds_path}')


## Полное обучение и финальный submission

Для полного train используется тот же общий признаковый каркас,
а финальный прогноз строится рекурсивно по тестовому горизонту.


In [ ]:
def fill_na_in_cat(df, feature_cols, cat_features_idx):
    """
    Заполняет NaN в категориальных колонках специальным токеном "__MISSING__".
    Это нужно, т.к. CatBoost не принимает NaN в cat_features. [web:472][web:475]
    """
    df = df.copy()
    cat_cols = [feature_cols[i] for i in cat_features_idx]
    for c in cat_cols:
        if c in df.columns:
            df[c] = df[c].astype("object").fillna("__MISSING__")
    return df


In [ ]:
def make_full_train_dataset(
    train_df,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Готовит полный датасет для обучения финальной CatBoost-модели
    и возвращает X_full, y_full, feature_cols, cat_features_idx, dates_aligned.
    """
    # Сортируем, чтобы лаги считались корректно
    df = train_df.sort_values(["store_nbr", "family", "date"]).copy()

    # Временные фичи
    df = add_time_features(df)

    # Лаги и rolling по sales
    df = add_lag_features(
        df,
        lags=lags,
        group_cols=["store_nbr", "family"],
        target_col="sales",
    )
    df = add_rolling_features(
        df,
        windows=roll_windows,
        group_cols=["store_nbr", "family"],
        target_col="sales",
    )

    # Все lag/rolling-фичи
    feature_cols_lag_roll = [
        c for c in df.columns
        if "lag_" in c or "rolling_mean_" in c
    ]

    # Отбрасываем строки без всех лагов/rolling
    df = df.dropna(subset=feature_cols_lag_roll)

    # Цель
    y_full = df["sales"].astype("float32")

    # Полный список фичей
    feature_cols = (
        NUM_FEATURES_BASE
        + TIME_FEATURES
        + feature_cols_lag_roll
        + BIN_FEATURES
        + CAT_FEATURES
    )
    feature_cols = [c for c in feature_cols if c in df.columns]

    X_full = df[feature_cols].copy()

    cat_features_idx = [
        i for i, c in enumerate(feature_cols)
        if c in (CAT_FEATURES + BIN_FEATURES)
    ]

    # Даты в том же порядке, что X_full / y_full
    dates_aligned = df["date"].values

    logger.info(
        f"[FULL] Dataset: X_full.shape={X_full.shape}, y_full.len={len(y_full)}, "
        f"n_cat_features={len(cat_features_idx)}"
    )

    return X_full, y_full, feature_cols, cat_features_idx, dates_aligned


In [ ]:
# Строим полный датасет
X_full, y_full, feature_cols_full, cat_features_idx_full, dates_full = make_full_train_dataset(
    train_processed,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
)

# Последняя дата (валидация на один день)
last_date = dates_full.max()
logger.info(f"[FULL] Validation date = {pd.to_datetime(last_date).date()}")

val_mask = dates_full == last_date
train_mask = dates_full < last_date

X_train_full = X_full[train_mask]
y_train_full = y_full[train_mask]

X_val_full = X_full[val_mask]
y_val_full = y_full[val_mask]

logger.info(
    f"[FULL] Train size = {X_train_full.shape}, val size (1 day) = {X_val_full.shape}"
)


In [ ]:
THREAD_COUNT = -1  

# Обрабатываем NaN в категориальных фичах
X_train_full_filled = fill_na_in_cat(X_train_full, feature_cols_full, cat_features_idx_full)
X_val_full_filled   = fill_na_in_cat(X_val_full, feature_cols_full, cat_features_idx_full)

y_train_log = np.log1p(y_train_full)
y_val_log   = np.log1p(y_val_full)

train_pool_full = Pool(
    data=X_train_full_filled,
    label=y_train_log,
    cat_features=cat_features_idx_full,
)
val_pool_full = Pool(
    data=X_val_full_filled,
    label=y_val_log,
    cat_features=cat_features_idx_full,
)

final_model = CatBoostRegressor(
    loss_function="RMSE",
    eval_metric="RMSE",
    depth=best_params["depth"],
    learning_rate=best_params["learning_rate"],
    l2_leaf_reg=best_params["l2_leaf_reg"],
    bagging_temperature=best_params["bagging_temperature"],
    border_count=best_params["border_count"],
    random_state=42,
    thread_count=THREAD_COUNT,
    task_type="CPU",
    verbose=False,
    od_type="Iter",
    od_wait=50,
    iterations=2000,
)

start_time = datetime.now()
logger.info(f"[FULL] Final CatBoost fit started at {start_time.strftime('%Y-%m-%d %H:%M:%S')}")

final_model.fit(
    train_pool_full,
    eval_set=val_pool_full,
    use_best_model=True,
    verbose=False,
)

end_time = datetime.now()
elapsed = (end_time - start_time).total_seconds()
logger.info(f"[FULL] Final CatBoost fit finished in {elapsed:.1f} seconds")


In [ ]:
def recursive_forecast_catboost_on_test(
    model,
    train_df,
    test_df,
    feature_cols,
    cat_features_idx,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
):
    """
    Рекурсивный 16-дневный прогноз на тесте:
    - история = весь train (с реальными sales, включая последний день);
    - для каждого дня теста:
        * добавляем его каркас (без sales) к истории,
        * пересчитываем фичи (time, lag, rolling),
        * предсказываем sales,
        * подставляем предсказания в историю, чтобы они участвовали в лагах
          на следующих днях;
    - в конце мержим прогнозы с test_df по (store_nbr, family, date),
      чтобы вернуть id и получить submission формата id,sales. [web:483][web:387]
    """
    # История: весь train
    hist = train_df.sort_values(["store_nbr", "family", "date"]).copy()

    # Тест: каркас (есть id, store_nbr, family, date и все exogenous-признаки)
    future = test_df.sort_values(["date", "store_nbr", "family"]).copy()

    test_dates = sorted(future["date"].unique())
    all_preds = []

    for cur_date in test_dates:
        cur_ts = pd.to_datetime(cur_date)
        logger.info(f"[TEST] Forecasting date {cur_ts.date()}")

        # Строки теста для текущей даты
        day_future = future[future["date"] == cur_date].copy()

        # Временный столбец sales=NaN, чтобы лаги считались только из hist
        day_future["sales"] = np.nan

        # История + текущий день
        combined = pd.concat([hist, day_future], axis=0, ignore_index=True)

        # Фичи времени + lag/rolling по sales
        combined = add_time_features(combined)
        combined = add_lag_features(
            combined,
            lags=lags,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )
        combined = add_rolling_features(
            combined,
            windows=roll_windows,
            group_cols=["store_nbr", "family"],
            target_col="sales",
        )

        # Берём только строки текущего дня уже с фичами
        cur_df = combined[combined["date"] == cur_date].copy()

        # ВАЖНО: на тесте НЕ делаем dropna по lag/rolling — CatBoost умеет с ними работать
        # cur_df = cur_df.dropna(subset=feature_cols_lag_roll)

        if cur_df.empty:
            logger.warning(f"[TEST] No rows to predict for {cur_ts.date()}, skipping")
            continue

        # X в том же порядке фичей, что при обучении
        X_cur = cur_df[feature_cols].copy()
        X_cur = fill_na_in_cat(X_cur, feature_cols, cat_features_idx)

        pool_cur = Pool(
            data=X_cur,
            cat_features=cat_features_idx,
        )

        # Предсказания в лог-пространстве → исходный масштаб
        y_pred_log = model.predict(pool_cur)
        y_pred = np.expm1(y_pred_log)
        y_pred = np.clip(y_pred, 0, None)

        # Сохраняем прогнозы вместе с ключами
        cur_df = cur_df[["store_nbr", "family", "date"]].copy()
        cur_df["sales"] = y_pred
        all_preds.append(cur_df)

        # Обновляем историю: на cur_date sales = предсказания (для будущих лагов)
        hist_update = day_future.copy()
        hist_update["sales"] = y_pred
        hist = pd.concat([hist, hist_update], axis=0, ignore_index=True)

    # Собираем все прогнозы по ключам
    preds_df = pd.concat(all_preds, axis=0, ignore_index=True)

    # Мерджим с test_df, чтобы вернуть id для сабмита
    submission_df = (
        test_df[["id", "store_nbr", "family", "date"]]
        .merge(
            preds_df,
            on=["store_nbr", "family", "date"],
            how="left",
        )
    )

    # На всякий случай проверим покрытие id
    missing_ids = set(test_df["id"]) - set(submission_df["id"])
    if missing_ids:
        logger.warning(f"[TEST] Missing predictions for {len(missing_ids)} ids")

    # Финальный формат для Kaggle
    submission_df = submission_df[["id", "sales"]]

    return submission_df


In [ ]:
submission_df = recursive_forecast_catboost_on_test(
    model=final_model,
    train_df=train_processed,
    test_df=test_processed,
    feature_cols=feature_cols_full,
    cat_features_idx=cat_features_idx_full,
    lags=(1, 7, 14, 28),
    roll_windows=(7, 28),
)
submission = submission_df.sort_values('id')[['id', 'sales']]
submission_path = SUBMISSIONS_DIR / 'submission_catboost_recursive.csv'
submission.to_csv(submission_path, index=False)
logger.info(f'CatBoost submission saved to {submission_path}')
submission.head()
